In [3]:
import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sqlite3
import os

In [4]:
conn = sqlite3.connect("world.db")
c = conn.cursor()

# distance テーブルを削除
c.execute("DROP TABLE IF EXISTS distance_point_3")
c.execute("CREATE TABLE IF NOT EXISTS distance_point_3 (hash TEXT, hash1 TEXT, hash2 TEXT, hash3 TEXT, distance REAL)")
c.execute("DROP TABLE IF EXISTS distance_point_4")
c.execute("CREATE TABLE IF NOT EXISTS distance_point_4 (hash TEXT, hash1 TEXT, hash2 TEXT, hash3 TEXT, distance REAL)")
c.close()

In [5]:
hash1 = "323"
hash2 = "525"
hash3 = "433"
hash4 = "336"

In [6]:
def generateAffineMatrix3D(matrix_affineA, matrix_affineAdash):
    # 各行列から平行移動を含む点を抽出
    points_A = np.array([mat[:3, 3] for mat in matrix_affineA])  # Aの各点 (x, y, z)
    points_Adash = np.array([mat[:3, 3] for mat in matrix_affineAdash])  # Adashの各点 (x', y', z')
    
    # アフィン変換行列を求めるための最小二乗法
    ones = np.ones((points_A.shape[0], 1))  # 点数分の1列
    P = np.hstack((points_A, ones))  # Aの点を拡張 [x, y, z, 1]
    print(P)
    Q = points_Adash  # Adashの点
    print(Q)
    
    # 最小二乗法で解を求める
    affine_matrix, _, _, _ = np.linalg.lstsq(P, Q, rcond=None)
    print("affine_matrix")
    print(affine_matrix)

    # 4x4のアフィン変換行列を構築
    affine_matrix_4x4 = np.eye(4)
    affine_matrix_4x4[:3, :] = affine_matrix.T
    
    return affine_matrix_4x4

def generateAffineMatrix3DSelfLU(matrix_affineA, matrix_affineAdash):
    """LU分解を用いた最小二乗法でアフィン変換行列を求める"""
    points_A = np.array([mat[:3, 3] for mat in matrix_affineA])
    points_Adash = np.array([mat[:3, 3] for mat in matrix_affineAdash]) 
    
    # 拡張行列を作成
    P = np.hstack((points_A, np.ones((points_A.shape[0], 1))))
    # print(P)
    Q = points_Adash
    # print(Q)

    # 最小二乗法で解を求める
    affine_matrix = leastSquaresMethodLU(P, Q)

    print("affine_matrix")
    print(affine_matrix)

    # 4x4のアフィン変換行列を構築
    affine_matrix = np.vstack((affine_matrix.T, np.array([0, 0, 0, 1])))

    return affine_matrix

def LU(A: np.array) -> np.array:
    """LU分解"""
    n = A.shape[0]
    L = np.eye(n)
    U = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            U[i, j] = A[i, j] - np.sum(L[i, :i] * U[:i, j])
        # print("U")
        # print(U)
        for j in range(i + 1, n):
            L[j, i] = (A[j, i] - np.sum(L[j, :i] * U[:i, i])) / U[i, i]
        # print("L")
        # print(L)

    return L, U

def eq_solve(P: np.array, Q: np.array) -> np.array:
    # LU分解
    L, U = LU(P)
    print("L")
    print(L)
    print("U")
    print(U)
    n, m = Q.shape  # Q の形状を取得
    y = np.zeros((n, m))  # 前進代入の解ベクトル y を用意

    # 前進代入
    for i in range(n):
        print("Q[i][k],dot",i)
        print(Q[i, :],np.dot(L[i, :i], y[:i, :]))
        y[i, :] = Q[i, :] - np.dot(L[i, :i], y[:i, :])

    print("y")
    print(y)

    x = np.zeros((n, m))  # 解ベクトル x を用意

    # 後退代入
    for i in range(n - 1, -1, -1):
        if np.abs(U[i, i]) < 1e-8:  # 0除算防止
            print(f"Warning: U[{i}, {i}] is nearly zero. Adding small value.")
            U[i, i] = 1e-8
        print(f"np.dot({U[i, i+1:]}, {x[i+1:, :]})")
        print(y[i, :],"\n",np.dot(U[i, i+1:], x[i+1:, :]),"\n",U[i, i])
        x[i, :] = (y[i, :] - np.dot(U[i, i+1:], x[i+1:, :])) / U[i, i]
        print(f"x {i}")
        print(x)

    print("x")
    print(x)
    return x

def leastSquaresMethodLU(P: np.array, Q: np.array) -> np.array:
    # 最小二乗法で解を求める
    affine_matrix = eq_solve(P.T @ P, P.T) @ Q
    print("eq_solve(P.T @ P, P.T)")
    print(eq_solve(P.T @ P, P.T))
    print("Q")
    print(Q)
    return affine_matrix

In [7]:
# データベースに接続
conn = sqlite3.connect("world.db")
c = conn.cursor()

# CREATE TABLE world (x INTEGER, y INTEGER, z INTEGER, hash TEXT, matrix_1_x REAL, matrix_1_y REAL, matrix_1_z REAL, matrix_2_x REAL, matrix_2_y REAL, matrix_2_z REAL)
# ワールドの座標系1のアフィン変換行列を取得
c.execute("SELECT * FROM world WHERE hash = ?", (hash1,))
rows1 = c.fetchall()
rows1 = [list(row) for row in rows1 ]

# ワールドの座標系2のアフィン変換行列を取得
c.execute("SELECT * FROM world WHERE hash = ?", (hash2,))
rows2 = c.fetchall()
rows2 = [list(row) for row in rows2]

# ワールドの座標系3のアフィン変換行列を取得
c.execute("SELECT * FROM world WHERE hash = ?", (hash3,))
rows3 = c.fetchall()
rows3 = [list(row) for row in rows3]

# ワールドの座標系4のアフィン変換行列を取得
c.execute("SELECT * FROM world WHERE hash = ?", (hash4,))
rows4 = c.fetchall()
rows4 = [list(row) for row in rows4]

c.close()

# ワールドの座標系1と座標系2のアフィン変換行列から座標系1と座標系3のアフィン変換行列を求める
matrix_affine1_point_3 = np.array([
    [[1, 0, 0, rows1[0][4]],[0, 1, 0, rows1[0][5]],[0, 0, 1, rows1[0][6]],[0, 0, 0, 1]],
    [[1, 0, 0, rows2[0][4]],[0, 1, 0, rows2[0][5]],[0, 0, 1, rows2[0][6]],[0, 0, 0, 1]],
    [[1, 0, 0, rows3[0][4]],[0, 1, 0, rows3[0][5]],[0, 0, 1, rows3[0][6]],[0, 0, 0, 1]],
])

matrix_affine2_point_3 = np.array([
    [[1, 0, 0, rows1[0][7]],[0, 1, 0, rows1[0][8]],[0, 0, 1, rows1[0][9]],[0, 0, 0, 1]],
    [[1, 0, 0, rows2[0][7]],[0, 1, 0, rows2[0][8]],[0, 0, 1, rows2[0][9]],[0, 0, 0, 1]],
    [[1, 0, 0, rows3[0][7]],[0, 1, 0, rows3[0][8]],[0, 0, 1, rows3[0][9]],[0, 0, 0, 1]],
])

matrix_affine1_point_4 = np.array([
    [[1, 0, 0, rows1[0][4]],[0, 1, 0, rows1[0][5]],[0, 0, 1, rows1[0][6]],[0, 0, 0, 1]],
    [[1, 0, 0, rows2[0][4]],[0, 1, 0, rows2[0][5]],[0, 0, 1, rows2[0][6]],[0, 0, 0, 1]],
    [[1, 0, 0, rows3[0][4]],[0, 1, 0, rows3[0][5]],[0, 0, 1, rows3[0][6]],[0, 0, 0, 1]],
    [[1, 0, 0, rows4[0][4]],[0, 1, 0, rows4[0][5]],[0, 0, 1, rows4[0][6]],[0, 0, 0, 1]],
])

matrix_affine2_point_4 = np.array([
    [[1, 0, 0, rows1[0][7]],[0, 1, 0, rows1[0][8]],[0, 0, 1, rows1[0][9]],[0, 0, 0, 1]],
    [[1, 0, 0, rows2[0][7]],[0, 1, 0, rows2[0][8]],[0, 0, 1, rows2[0][9]],[0, 0, 0, 1]],
    [[1, 0, 0, rows3[0][7]],[0, 1, 0, rows3[0][8]],[0, 0, 1, rows3[0][9]],[0, 0, 0, 1]],
    [[1, 0, 0, rows4[0][7]],[0, 1, 0, rows4[0][8]],[0, 0, 1, rows4[0][9]],[0, 0, 0, 1]],
])

In [8]:
def generateAffineMatrix3DSelfLUAndInsertDistance(points: int,matrix_affine1: np.array, matrix_affine2: np.array):
    affine_self_LU = generateAffineMatrix3DSelfLU(matrix_affine1, matrix_affine2)
    print(affine_self_LU)

    conn = sqlite3.connect("world.db")
    c = conn.cursor()

    hash = "000"

    # 5個飛びで全ての座標を計算
    for i in range(0, 50, 5): 
        for j in range(0, 50, 5):
            for k in range(0, 50, 5):
                hash = str(i) + str(j) + str(k)
                c.execute("SELECT * FROM world WHERE hash = ?", (hash,))
                rows1 = c.fetchall()
                rows1 = [list(row) for row in rows1 ]
                if len(rows1) == 0:
                    continue
                matrix = np.array([
                    rows1[0][4],
                    rows1[0][5],
                    rows1[0][6],
                    1
                ])

                affine_self_LU_times_matrix = np.dot(affine_self_LU, matrix)

                # ユークリッド距離を求める
                distance = math.sqrt((affine_self_LU_times_matrix[0] - rows1[0][7]) ** 2 + (affine_self_LU_times_matrix[1] - rows1[0][8]) ** 2 + (affine_self_LU_times_matrix[2] - rows1[0][9]) ** 2)

                c.execute(f"INSERT INTO distance_point_{points} (hash, hash1, hash2, hash3, distance) VALUES (?, ?, ?, ?, ?)", (hash, hash1, hash2, hash3, distance))
                conn.commit()

    c.close()

generateAffineMatrix3DSelfLUAndInsertDistance(3,matrix_affine1_point_3, matrix_affine2_point_3)
generateAffineMatrix3DSelfLUAndInsertDistance(4,matrix_affine1_point_4, matrix_affine2_point_4)

L
[[1.         0.         0.         0.        ]
 [0.90721649 1.         0.         0.        ]
 [2.93814433 1.23893805 1.         0.        ]
 [0.12371134 0.09734513 0.04347826 1.        ]]
U
[[1.94000000e+02 1.76000000e+02 5.70000000e+02 2.40000000e+01]
 [0.00000000e+00 2.32989691e+00 2.88659794e+00 2.26804124e-01]
 [0.00000000e+00 0.00000000e+00 4.68141593e+00 2.03539823e-01]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]]
Q[i][k],dot 0
[7. 9. 8.] [0. 0. 0.]
Q[i][k],dot 1
[7. 7. 8.] [6.35051546 8.16494845 7.25773196]
Q[i][k],dot 2
[23. 25. 23.] [21.37168142 25.         24.42477876]
Q[i][k],dot 3
[1. 1. 1.] [1. 1. 1.]
y
[[ 7.00000000e+00  9.00000000e+00  8.00000000e+00]
 [ 6.49484536e-01 -1.16494845e+00  7.42268041e-01]
 [ 1.62831858e+00 -2.13162821e-14 -1.42477876e+00]
 [ 8.88178420e-16  4.44089210e-16 -1.11022302e-15]]
np.dot([], [])
[ 8.88178420e-16  4.44089210e-16 -1.11022302e-15] 
 [0. 0. 0.] 
 1e-08
x 3
[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.000